# Overfitting Simulation: The Bias-Variance Tradeoff in Action

**HPM 883: Advanced Quantitative Methods | Session 2.1**

In this notebook, you'll:
1. Generate noisy data from a known function (sin wave)
2. Fit polynomials of increasing complexity
3. **See overfitting in action** — why training error alone is misleading
4. Use cross-validation to find the right model complexity

## Setup

In [ ]:
# Import required packages
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import make_pipeline

plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline

print("Packages loaded successfully!")

## Part 1: Generate Noisy Data from a Known Function

We'll generate data from $y = \sin(x) + \varepsilon$, where $\varepsilon \sim N(0, 0.3)$.

**Key questions:**
- When does the model fit too much noise?
- Why does training error alone mislead us?
- How do we find the right complexity?

In [ ]:
# Generate population and sample
np.random.seed(42)

n_population = 200  # Full population
n_sample = 30       # Our training sample

# Generate population data
X_pop = np.sort(np.random.uniform(0, 2 * np.pi, n_population))
y_true_pop = np.sin(X_pop)
y_pop = y_true_pop + np.random.normal(0, 0.3, n_population)

# Take a random sample for training
sample_idx = np.sort(np.random.choice(n_population, n_sample, replace=False))
X_sample = X_pop[sample_idx]
y_sample = y_pop[sample_idx]

# The unseen population points (for testing)
unseen_idx = np.setdiff1d(np.arange(n_population), sample_idx)
X_unseen = X_pop[unseen_idx]
y_unseen = y_pop[unseen_idx]

print(f"Population: {n_population} points")
print(f"Sample (training): {n_sample} points")
print(f"Unseen (testing): {len(unseen_idx)} points")

In [ ]:
# Visualize population vs sample
x_smooth = np.linspace(0, 2 * np.pi, 100)

fig, ax = plt.subplots(figsize=(12, 5))

ax.scatter(X_unseen, y_unseen, s=40, c='lightgray', alpha=0.5,
           label=f'Unseen population ({len(unseen_idx)} points)')
ax.scatter(X_sample, y_sample, s=80, c='blue', edgecolors='black',
           zorder=5, label=f'Training sample ({n_sample} points)')
ax.plot(x_smooth, np.sin(x_smooth), 'r--', linewidth=2, alpha=0.7,
        label='True: sin(x)')

ax.set_xlabel('x', fontsize=12)
ax.set_ylabel('y', fontsize=12)
ax.set_title('Population vs Sample: We Only See the Blue Points!', fontsize=14)
ax.legend()
plt.tight_layout()
plt.show()

print("\nWe train on the BLUE points, but want to predict the GRAY points well!")

## Part 2: Fitting Polynomials of Increasing Degree

Let's fit polynomials with degrees 1 (linear), 3, 10, and 20 to see the progression from underfitting to overfitting.

In [ ]:
# Fit and visualize polynomial models
colors = {1: 'green', 3: 'orange', 10: 'purple', 20: 'red'}
fitted_models = {}

for degree in [1, 3, 10, 20]:
    poly = PolynomialFeatures(degree)
    X_poly = poly.fit_transform(X_sample.reshape(-1, 1))
    model = LinearRegression()
    model.fit(X_poly, y_sample)
    
    y_pred_train = model.predict(X_poly)
    train_mse = mean_squared_error(y_sample, y_pred_train)
    
    X_smooth_poly = poly.transform(x_smooth.reshape(-1, 1))
    y_pred_smooth = model.predict(X_smooth_poly)
    fitted_models[degree] = {'predictions': y_pred_smooth, 'mse': train_mse}

# Plot all fits together
fig, ax = plt.subplots(figsize=(12, 6))

ax.scatter(X_unseen, y_unseen, s=30, c='lightgray', alpha=0.3, label='Unseen population')
ax.scatter(X_sample, y_sample, s=60, c='blue', edgecolors='black', zorder=5, label='Training sample')
ax.plot(x_smooth, np.sin(x_smooth), 'r--', linewidth=2, alpha=0.5, label='True: sin(x)')

for d in sorted(fitted_models.keys()):
    info = fitted_models[d]
    ax.plot(x_smooth, info['predictions'], color=colors[d], linewidth=2, 
            label=f'Degree {d} (Train MSE: {info["mse"]:.3f})')

ax.set_xlabel('x', fontsize=12)
ax.set_ylabel('y', fontsize=12)
ax.set_title('Polynomial Fits: From Underfitting to Overfitting', fontsize=14)
ax.legend(loc='upper right')
ax.set_ylim(-2, 2)
plt.tight_layout()
plt.show()

print("\nNotice: Training MSE keeps decreasing... but are higher-degree models actually better?")

## Part 3: Sample Error vs Population Error

Now let's reveal the truth: how do these models perform on **unseen** data?

In [ ]:
# Compare sample error vs population error
degrees_to_compare = [1, 3, 10, 20]
results = []

for degree in degrees_to_compare:
    poly = PolynomialFeatures(degree)
    X_sample_poly = poly.fit_transform(X_sample.reshape(-1, 1))
    model = LinearRegression()
    model.fit(X_sample_poly, y_sample)

    y_sample_pred = model.predict(X_sample_poly)
    sample_mse = mean_squared_error(y_sample, y_sample_pred)

    X_unseen_poly = poly.transform(X_unseen.reshape(-1, 1))
    y_unseen_pred = model.predict(X_unseen_poly)
    unseen_mse = mean_squared_error(y_unseen, y_unseen_pred)

    results.append({'Degree': degree, 'Sample MSE': round(sample_mse, 4), 'Unseen MSE': round(unseen_mse, 4)})

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

In [ ]:
# Visualize the overfitting gap
fig, ax = plt.subplots(figsize=(10, 6))

x_pos = np.arange(len(degrees_to_compare))
width = 0.35

bars1 = ax.bar(x_pos - width/2, results_df['Sample MSE'], width,
               label='Sample (Training) MSE', color='blue', alpha=0.7)
bars2 = ax.bar(x_pos + width/2, results_df['Unseen MSE'], width,
               label='Unseen (Test) MSE', color='red', alpha=0.7)

ax.set_xlabel('Polynomial Degree', fontsize=12)
ax.set_ylabel('Mean Squared Error', fontsize=12)
ax.set_title('The Overfitting Gap: Sample Error vs Population Error', fontsize=14)
ax.set_xticks(x_pos)
ax.set_xticklabels(degrees_to_compare)
ax.legend()

plt.tight_layout()
plt.show()

print("\nKey insight: As degree increases, sample error drops but unseen error explodes!")
print("The gap between blue and red bars is the 'overfitting penalty'.")

## Part 4: Train vs Test Error Across All Degrees

Let's systematically compare train vs test error to see the bias-variance tradeoff.

In [ ]:
# Split sample data for train/test evaluation
X_train, X_test, y_train, y_test = train_test_split(
    X_sample, y_sample, test_size=0.3, random_state=42
)

degrees_to_test = range(1, 16)
train_errors = []
test_errors = []

for degree in degrees_to_test:
    poly = PolynomialFeatures(degree)
    X_train_poly = poly.fit_transform(X_train.reshape(-1, 1))
    X_test_poly = poly.transform(X_test.reshape(-1, 1))
    
    model = LinearRegression()
    model.fit(X_train_poly, y_train)
    
    train_errors.append(mean_squared_error(y_train, model.predict(X_train_poly)))
    test_errors.append(mean_squared_error(y_test, model.predict(X_test_poly)))

# Plot
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(degrees_to_test, train_errors, 'b-o', linewidth=2, markersize=8, label='Training Error')
ax.plot(degrees_to_test, test_errors, 'r-s', linewidth=2, markersize=8, label='Test Error')

optimal_degree = list(degrees_to_test)[np.argmin(test_errors)]
ax.axvline(x=optimal_degree, color='green', linestyle='--', alpha=0.7, 
           label=f'Optimal: Degree {optimal_degree}')

ax.set_xlabel('Polynomial Degree', fontsize=12)
ax.set_ylabel('Mean Squared Error', fontsize=12)
ax.set_title('The Bias-Variance Tradeoff in Action', fontsize=14)
ax.legend()
ax.set_yscale('log')
plt.tight_layout()
plt.show()

print(f"\nTraining error keeps decreasing, but test error rises after degree {optimal_degree}!")
print("This is the bias-variance tradeoff: more complexity eventually hurts generalization.")

## Part 5: Cross-Validation to Select Model Complexity

Instead of a single train/test split, let's use **K-fold cross-validation** for a more stable estimate.

In [ ]:
# Cross-validation to find optimal degree
degrees_to_test = range(1, 12)
cv_scores = []

for degree in degrees_to_test:
    pipeline = make_pipeline(
        PolynomialFeatures(degree),
        LinearRegression()
    )
    
    scores = cross_val_score(pipeline, X_sample.reshape(-1, 1), y_sample, 
                            cv=5, scoring='neg_mean_squared_error')
    cv_scores.append(-scores.mean())

# Plot
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(degrees_to_test, cv_scores, 'g-o', linewidth=2, markersize=8)

optimal_cv_degree = list(degrees_to_test)[np.argmin(cv_scores)]
ax.axvline(x=optimal_cv_degree, color='red', linestyle='--', alpha=0.7,
           label=f'CV Optimal: Degree {optimal_cv_degree}')

ax.set_xlabel('Polynomial Degree', fontsize=12)
ax.set_ylabel('Cross-Validation MSE', fontsize=12)
ax.set_title('Using Cross-Validation to Select Model Complexity', fontsize=14)
ax.legend()
plt.tight_layout()
plt.show()

print(f"\nCross-validation suggests degree {optimal_cv_degree} is optimal!")

## Key Takeaways

1. **We only see a sample** of the true population
2. **Training error** measures fit on the sample — it always decreases with complexity
3. **Test/population error** measures generalization to unseen data
4. **Overfitting** = great on sample, terrible on population
5. **Cross-validation** provides a more stable estimate of the right complexity

---

**Connection to lecture**: This is why we need regularization (LASSO, Ridge) — instead of choosing polynomial degree, we add a penalty that automatically constrains model complexity.

```
This notebook:  Choose degree to control complexity
Regularization: Use penalty parameter (λ) to control complexity
```

The same principle applies: **constrain complexity to prevent overfitting**.